In [ ]:
import yaml
import os
import shutil
import copy
import subprocess
import numpy as np

In [ ]:
bar_1_template = {
    'type': 'single',
    'input mesh file': 'bar-1.g',
    'output mesh file': 'bar-1.e',
    'model': {
        'type': 'solid mechanics',
        'material': {
            'blocks': {
                'fine': 'elastic'
            },
            'elastic': {
                'model': 'linear elastic',
                'elastic modulus': None,
                "Poisson's ratio": 0.25,
                'density': 1000.0
            }
        }
    },
    'time integrator': None,
    'initial conditions': {
        'displacement': [
            {
                'node set': 'nsall',
                'component': 'x',
                'function': '{replace}' #str("\"5.1359 * t\"")
            }
        ]
    },
    'boundary conditions': {
        'Schwarz contact': [
            {
                'side set': 'ssx+',
                'source': 'bar-2.yaml',
                'source block': 'coarse',
                'source side set': 'ssx-',
                'friction type': 'frictionless',
                'swap BC types': 'false'
            }
        ]
    },
    'solver': None
}

bar_2_template = {
    'type': 'single',
    'input mesh file': 'bar-2.g',
    'output mesh file': 'bar-2.e',
    'model': {
        'type': 'solid mechanics',
        'material': {
            'blocks': {
                'coarse': 'elastic'
            },
            'elastic': {
                'model': 'linear elastic',
                'elastic modulus': None,
                "Poisson's ratio": 0.25,
                'density': 1000.0
            }
        }
    },
    'time integrator': None,
    'initial conditions': {
        'displacement': [
            {
                'node set': 'nsall',
                'component': 'x',
                'function': '{-replace}'
            }
        ]
    },
    'boundary conditions': {
        'Schwarz contact': [
            {
                'side set': 'ssx-',
                'source': 'bar-1.yaml',
                'source block': 'fine',
                'source side set': 'ssx+',
                'friction type': 'frictionless',
                'swap BC types': 'false'
            }
        ]
    },
    'solver': None
}

solver_templates = [
    {
        'type': 'Hessian minimizer',
        'step': 'full Newton',
        'minimum iterations': 1,
        'maximum iterations': 16,
        'relative tolerance': 1.0e-10,
        'absolute tolerance': 1.0e-06
    },
    {
        'type': 'explicit solver',
        'step': 'explicit'
    }
]

integrator_templates = [
    {
        'type': 'Newmark',
        'β': 0.25,
        'γ': 0.5
    },
    {
        'type': 'central difference',
        'CFL': 0.1,
        'γ': 0.5
    }
]

schwarz_main_template = {
    'type': 'multi',
    'domains': ['bar-1.yaml', 'bar-2.yaml'],
    'Exodus output interval': 1,
    'CSV output interval': None,
    'initial time': None,
    'time step': None,
    'final time': None,
    'relaxation parameter': 1.0,
    'naive stabilized': False,
    'minimum iterations': 1,
    'maximum iterations': 16,
    'relative tolerance': 1.0e-08,
    'absolute tolerance': 1.0e-07
}


In [ ]:
def replace_placeholders_in_file(filename, velocity):
    # Read the content of the file
    with open(filename, 'r', encoding='utf-8') as file:
        content = file.read()

    # Replace placeholders with the desired strings
    content = content.replace("'{replace}'", f'\"{velocity} * t\"')
    content = content.replace("'{-replace}'", f'\"-{velocity} * t\"')

    # Write the modified content back to the file
    with open(filename, 'w', encoding='utf-8') as file:
        file.write(content)

    print(f"Replacements made in '{filename}' successfully.")

In [ ]:
# Define the cases, velocities, and elastic moduli

velocities = [1, 10, 100]  # Velocities
elastic_moduli = [10**9, 10**11, 10**13]  # Elastic moduli

# Create the dictionary
case_velocity_modulus_map = {}

case = 0
for velocity in velocities:
    for modulus in elastic_moduli:
        key = case
        case_velocity_modulus_map[key] =  [velocity, modulus]
        case +=1 
# Display the resulting dictionary
for key, value in case_velocity_modulus_map.items():
    print(f"{key}: {value}")

In [ ]:

def create_cases(mesh_folder):
    pwd = os.getcwd()



    for key, value in case_velocity_modulus_map.items():
        # Check if the output folder exists and delete it if it does
        case_folder = f'case_{key}'
        if os.path.exists(case_folder):
            #shutil.rmtree(case_folder)
            continue
        # Create the output folder
        os.makedirs(case_folder, exist_ok=True)
        # Copy meshes 
        shutil.copy(f'{mesh_folder}/bar-1.g', case_folder)
        shutil.copy(f'{mesh_folder}/bar-2.g', case_folder)

        velocity = value[0]
        E_mod = value[1]

        bar_1 = copy.copy(bar_1_template)
        bar_1['time integrator'] = integrator_templates[0]
        bar_1['solver'] = solver_templates[0]
        bar_1['boundary conditions']['Schwarz contact'][0]['swap BC types'] = False
        bar_1['model']['material']['elastic']['elastic modulus'] = E_mod

        with open(f'{case_folder}/bar-1.yaml', 'w',  encoding='utf-8') as file:
            yaml.dump(bar_1, file, default_flow_style=False, allow_unicode=True)

        bar_2 = copy.copy(bar_2_template)
        bar_2['time integrator'] = integrator_templates[0]
        bar_2['solver'] = solver_templates[0]
        bar_2['boundary conditions']['Schwarz contact'][0]['swap BC types'] = False
        bar_2['model']['material']['elastic']['elastic modulus'] = E_mod

        with open(f'{case_folder}/bar-2.yaml', 'w', encoding='utf-8') as file:
            yaml.dump(bar_2, file, default_flow_style=False, allow_unicode=True)


        
        replace_placeholders_in_file(f'{case_folder}/bar-1.yaml', velocity)
        replace_placeholders_in_file(f'{case_folder}/bar-2.yaml', velocity)

        main_yaml = copy.copy(schwarz_main_template)
        ts = float(1e-4/2 * np.sqrt(2) / np.sqrt(E_mod/1.e3))
        main_yaml['time step'] = ts
        main_yaml['initial time'] = -1e-6 * 100/velocity
        main_yaml['final time'] = 3e-6 * 100/velocity
        time_steps = (main_yaml['final time'] - main_yaml['initial time'])/main_yaml['time step']
        # Write 1000 files
        csv_output_int = int(time_steps/1000)
        if csv_output_int < 1:
            csv_output_int = 1
        main_yaml['CSV output interval'] = csv_output_int
        
        with open(f'{case_folder}/bars.yaml', 'w', encoding='utf-8') as file:
            yaml.dump(main_yaml, file, default_flow_style=False, allow_unicode=True)

        os.chdir(case_folder)
        command = [
                'julia',
                '--project=@.',
                '/Users/bphung/Work/software/Norma.jl/src/Norma.jl',
                'bars.yaml'
            ]
        # Run the command
        success = None
        try:
            subprocess.run(command, check=True)
            success = True
        except:
            success = False
        os.chdir(pwd)
        if success == False:
            shutil.rmtree(case_folder)
        
create_cases('mesh_1')